In [1]:
# Necessary Installations:
!pip -q install kagglehub transformers

In [2]:
# (1) Dataset download for ViVit and CNN14
import kagglehub
import os
import shutil

# 2000 Videos: 1k Violence, 1k non-violence
rlv_path = kagglehub.dataset_download("mohamedmustafa/real-life-violence-situations-dataset")
print(f"\nBaseline Dataset saved successfully at: {rlv_path}")
print(os.listdir(rlv_path))

# Downloading shooting footage from USF-Crime dataset

# Create a clean folder in Colab to store just these videos
shooting_target_dir = "/content/shooting_videos"
os.makedirs(shooting_target_dir, exist_ok=True)

download_count = 0
for i in range(1, 60):
    file_name = f"Shooting{i:03d}_x264.mp4"
    target_file = f"UCF_Crimes/UCF_Crimes/Videos/Shooting/{file_name}"
    try:
        # Download strictly the single file to bypass the folder crash
        file_path = kagglehub.dataset_download(
            "vigneshwar472/ucaucf-crime-annotation-dataset",
            path=target_file
        )
        shutil.copy(file_path, os.path.join(shooting_target_dir, file_name))
        download_count += 1
        print(f"Downloaded: {file_name}")

    except Exception:
        pass

print(f"\nSuccess! Successfully extracted {download_count} shooting videos.")
print("Files are safely located at:", shooting_target_dir)

KeyboardInterrupt: 

In [ ]:
# Dataset Video Processor
import os
import cv2
import torch
import numpy as np
import kagglehub
from tqdm import tqdm

base_rlv_path = kagglehub.dataset_download("mohamedmustafa/real-life-violence-situations-dataset")
rlv_path = os.path.join(base_rlv_path, "Real Life Violence Dataset")
shooting_path = "/content/shooting_videos"

tensor_out_dir = "/content/fast_tensors_5fps"
os.makedirs(os.path.join(tensor_out_dir, "Violent"), exist_ok=True)
os.makedirs(os.path.join(tensor_out_dir, "Safe"), exist_ok=True)

video_data = []

for file in os.listdir(shooting_path):
    if file.endswith('.mp4'):
        video_data.append((os.path.join(shooting_path, file), 1, "Violent"))

for root, dirs, files in os.walk(rlv_path):
    for file in files:
        if file.endswith('.mp4'):
            full_path = os.path.join(root, file)
            if 'NonViolence' in full_path:
                video_data.append((full_path, 0, "Safe"))
            else:
                video_data.append((full_path, 1, "Violent"))

print(f"Starting Storage-Optimized extraction for {len(video_data)} videos...")

def process_and_chunk_video(video_path, base_save_path, target_fps=5, frames_per_chunk=32, target_size=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    original_fps = cap.get(cv2.CAP_PROP_FPS)

    if original_fps <= 0:
        original_fps = 30.0

    frame_indices = []
    current_frame = 0.0
    step = original_fps / target_fps

    while current_frame < total_frames:
        frame_indices.append(int(current_frame))
        current_frame += step

    chunks = [frame_indices[i:i + frames_per_chunk] for i in range(0, len(frame_indices), frames_per_chunk)]

    for chunk_idx, chunk in enumerate(chunks):
        frames = []
        for idx in chunk:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            success, frame = cap.read()
            if success:
                frame = cv2.resize(frame, target_size)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)

        while len(frames) < frames_per_chunk:
            frames.append(np.zeros((*target_size, 3), dtype=np.uint8))

        frames = np.array(frames, dtype=np.uint8)
        frames_tensor = torch.tensor(frames).permute(3, 0, 1, 2)

        chunk_save_path = base_save_path.replace(".pt", f"_chunk{chunk_idx}.pt")
        if not os.path.exists(chunk_save_path):
            torch.save(frames_tensor, chunk_save_path)

    cap.release()

for video_path, label, folder_name in tqdm(video_data, desc="Chunking Videos to Tensors"):
    base_name = os.path.basename(video_path).replace(".mp4", ".pt")
    base_save_path = os.path.join(tensor_out_dir, folder_name, base_name)
    process_and_chunk_video(video_path, base_save_path)

print(f"\nSuccess! Optimized tensors safely stored at {tensor_out_dir}")

Using Colab cache for faster access to the 'real-life-violence-situations-dataset' dataset.
Starting Storage-Optimized extraction for 2001 videos...


Chunking Videos to Tensors: 100%|██████████| 2001/2001 [1:21:20<00:00,  2.44s/it]


Success! Optimized tensors safely stored at /content/fast_tensors_5fps


In [ ]:
# BACKUP: Save Dataset to Google Drive

import shutil
import os
from google.colab import drive

drive.mount('/content/drive')

source_dir = "/content/fast_tensors_5fps"
drive_backup_path = "/content/drive/MyDrive/ViViT_Dataset_5FPS_Backup"
shutil.make_archive(drive_backup_path, 'zip', source_dir)

Mounted at /content/drive


'/content/drive/MyDrive/ViViT_Dataset_5FPS_Backup.zip'

In [3]:
# RESTORE: Load Dataset from Google Drive
import shutil
import os
from google.colab import drive

drive.mount('/content/drive')
backup_zip_path = "/content/drive/MyDrive/ViViT_Dataset_5FPS_Backup.zip"
extract_target_dir = "/content/fast_tensors_5fps"

if os.path.exists(backup_zip_path):
    shutil.unpack_archive(backup_zip_path, extract_target_dir)
    print("Success")
else:
    print("ERROR: Could not find the backup zip file in your Google Drive. Double-check the file name.")

Mounted at /content/drive
Success


In [4]:
# Data Loader
import os
import torch
from torch.utils.data import Dataset, DataLoader

tensor_dir = "/content/fast_tensors_5fps"

class FastViViTDataset(Dataset):
    def __init__(self, tensor_dir):
        self.tensor_paths = []
        self.labels = []

        for label_name, label_val in [("Safe", 0), ("Violent", 1)]:
            folder_path = os.path.join(tensor_dir, label_name)
            if os.path.exists(folder_path):
                for file in os.listdir(folder_path):
                    if file.endswith('.pt'):
                        self.tensor_paths.append(os.path.join(folder_path, file))
                        self.labels.append(label_val)

    def __len__(self):
        return len(self.tensor_paths)

    def __getitem__(self, idx):
        tensor_path = self.tensor_paths[idx]
        label = self.labels[idx]

        video_tensor = torch.load(tensor_path).float() / 255.0
        label_tensor = torch.tensor(label, dtype=torch.long)

        return video_tensor, label_tensor

fast_dataset = FastViViTDataset(tensor_dir)

train_loader = DataLoader(
    fast_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2
)

print(f"DataLoader ready! Mapped {len(fast_dataset)} chunked tensors.")

DataLoader ready! Mapped 2873 chunked tensors.


In [5]:
# (3) Download pre-trained model, freeze body, swap head for Violent/Safe
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import VivitForVideoClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Downloading Google's ViViT brain...")
model = VivitForVideoClassification.from_pretrained(
    "google/vivit-b-16x2-kinetics400",
    ignore_mismatched_sizes=True
)

# Brain surgery: Swap the 400-class head for our custom 2-class head
in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, 2)
print(f"Surgery complete! Model now outputs {model.classifier.out_features} classes (Violent/Safe).\n")

# Freeze body:
for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

model = model.to(device)

# Setup Optimizer and Loss Function
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print("Model architecture built and ready for training!")

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/356M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Surgery complete! Model now outputs 2 classes (Violent/Safe).

Model architecture built and ready for training!


In [6]:
# (3) Build the ViViT Architecture
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import VivitForVideoClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Downloading fresh Google ViViT brain...")
model = VivitForVideoClassification.from_pretrained(
    "google/vivit-b-16x2-kinetics400",
    ignore_mismatched_sizes=True
)

in_features = model.classifier.in_features
model.classifier = nn.Linear(in_features, 2)

for name, param in model.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

model = model.to(device)

optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()
print("Model built and ready for 5-FPS training!")

Using device: cpu


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model built and ready for 5-FPS training!


In [7]:
# (4) ViViT 5-FPS Training Loop
import time
import os
from tqdm import tqdm
from google.colab import drive

print("Connecting to Google Drive...")
drive.mount('/content/drive')

# File path
save_dir = '/content/drive/MyDrive/ViViT_5FPS_Weights'
os.makedirs(save_dir, exist_ok=True)
print(f"Checkpoints will be safely saved to: {save_dir}\n")

epochs = 5
print("Starting ViViT Training on 5-FPS Chunks...")

scaler = torch.amp.GradScaler('cuda')

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_predictions = 0
    start_time = time.time()

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

    for batch_idx, (videos, labels) in enumerate(progress_bar):
        # Shape the data perfectly for the ViViT model
        videos = videos.permute(0, 2, 1, 3, 4).to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs = model(videos)
            logits = outputs.logits
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

        progress_bar.set_postfix({'Loss': f"{loss.item():.4f}"})

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = (correct_predictions / total_predictions) * 100
    elapsed_time = time.time() - start_time

    print(f"\n--- Epoch {epoch+1} Completed in {elapsed_time:.2f}s ---")
    print(f"Average Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

    checkpoint_path = os.path.join(save_dir, f"vivit_5fps_epoch_{epoch+1}_checkpoint.pt")
    torch.save(model.state_dict(), checkpoint_path)
    print(f"Progress Saved! Checkpoint uploaded to Google Drive: {checkpoint_path}\n")

final_path = os.path.join(save_dir, "custom_vivit_5fps_FINAL.pt")
torch.save(model.state_dict(), final_path)
print(f"Training Complete! Final 5-FPS model weights safely stored at: {final_path}")

Connecting to Google Drive...


/tmp/ipykernel_15583/557696432.py:18: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  scaler = torch.amp.GradScaler('cuda')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints will be safely saved to: /content/drive/MyDrive/ViViT_5FPS_Weights

Starting ViViT Training on 5-FPS Chunks...


Epoch 1/5:   0%|          | 0/719 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_15583/557696432.py:36: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  with torch.amp.autocast('cuda'):
Epoch 1/5:   0%|          | 1/719 [02:27<29:27:42, 147.72s/it, Loss=0.7871]


KeyboardInterrupt: 